# **4x4 Grid Example for Contiguity, Adjacency and Spatial Weight Matrices**



- $A$ encodes who is a neighbor (rook vs queen),
- $D$ counts how many neighbors each cell has,
- $W=D^{−1} A$ turns sums into averages,
- $L=D−A$ measures difference from neighbors
- Applying $A$ or $W$ to $vec(X)$ matches the familiar stencil computations on the grid


In [ ]:
# Tiny 4×4 example for Rook & Queen: build A_rook, A_queen, D, W, L, and verify W x equals raster stencil averages.

import numpy as np

# ----- Helpers -----
def chain_adj(k: int) -> np.ndarray:
    """1D chain adjacency (no wrap)."""
    A = np.zeros((k,k), dtype=int)
    for i in range(k-1):
        A[i, i+1] = 1
        A[i+1, i] = 1
    return A

def row_standardize(A: np.ndarray) -> np.ndarray:
    deg = A.sum(axis=1, keepdims=True).astype(float)
    W = np.divide(A, deg, out=np.zeros_like(A, dtype=float), where=deg>0)
    return W

def print_mat(name, M, fmt="{:>6.2f}", max_rows=16, max_cols=16):
    print(f"\n{name} (shape={M.shape}):")
    # format numeric for compact readability
    if M.dtype.kind in "iu":
        fmt = "{:>3d}"
    rows, cols = M.shape
    for i in range(min(rows, max_rows)):
        row = []
        for j in range(min(cols, max_cols)):
            row.append(fmt.format(M[i,j]))
        print(" ".join(row))
    if rows>max_rows or cols>max_cols:
        print("...")

# ----- Grid size -----
m, n = 4, 4           # 4×4 grid
N = m * n

# 1D pieces
Cm = chain_adj(m)     # vertical (rows)
Cn = chain_adj(n)     # horizontal (cols)
Im = np.eye(m, dtype=int)
In = np.eye(n, dtype=int)

# Kronecker constructions
A_rook  = np.kron(In, Cm) + np.kron(Cn, Im)
A_queen = A_rook + np.kron(Cn, Cm)

# Degree, weights, Laplacian (for rook)
deg_rook  = A_rook.sum(axis=1)
D_rook    = np.diag(deg_rook)
W_rook    = row_standardize(A_rook)
L_rook    = D_rook - A_rook

# Degree, weights, Laplacian (for queen)
deg_queen = A_queen.sum(axis=1)
D_queen   = np.diag(deg_queen)
W_queen   = row_standardize(A_queen)
L_queen   = D_queen - A_queen

# ----- Toy raster X (values 1..16), vectorized x -----
X = np.arange(1, N+1).reshape(m, n)  # 1..16 laid out row-major
x = X.ravel(order="C")               # vec(X)

# ----- Apply via big matrices vs 2D stencils -----
# Rook stencil sum on grid: Cm @ X + X @ Cn^T
rook_sum_grid = Cm @ X + X @ Cn.T
rook_sum_vec  = rook_sum_grid.ravel(order="C")
rook_sum_vec_fromA = A_rook @ x

# Rook average on grid: divide by degree per cell
rook_deg_grid = deg_rook.reshape(m, n)
rook_avg_grid = np.divide(rook_sum_grid, rook_deg_grid, where=rook_deg_grid>0)
rook_avg_vec  = rook_avg_grid.ravel(order="C")
rook_avg_vec_fromW = W_rook @ x

# Queen stencil sum on grid: Cm @ X + X @ Cn^T + Cm @ X @ Cn^T
queen_sum_grid = Cm @ X + X @ Cn.T + Cm @ X @ Cn.T
queen_sum_vec  = queen_sum_grid.ravel(order="C")
queen_sum_vec_fromA = A_queen @ x

queen_deg_grid = deg_queen.reshape(m, n)
queen_avg_grid = np.divide(queen_sum_grid, queen_deg_grid, where=queen_deg_grid>0)
queen_avg_vec  = queen_avg_grid.ravel(order="C")
queen_avg_vec_fromW = W_queen @ x

# ----- Print matrices -----
print_mat("A_rook (adjacency, 4-neighbors)", A_rook, fmt="{:>2d}")
print_mat("A_queen (adjacency, 8-neighbors)", A_queen, fmt="{:>2d}")
print_mat("D_rook (degree diag)", D_rook, fmt="{:>2d}")
print_mat("W_rook (row-standardized)", W_rook, fmt="{:>6.2f}")
print_mat("L_rook (D - A)", L_rook, fmt="{:>2d}")
print_mat("D_queen (degree diag)", D_queen, fmt="{:>2d}")
print_mat("W_queen (row-standardized)", W_queen, fmt="{:>6.2f}")
print_mat("L_queen (D - A)", L_queen, fmt="{:>2d}")

# ----- Verify equality statements -----
print("\nVerification (Rook):")
print("A_rook @ x equals rook_sum_vec? ", np.allclose(rook_sum_vec_fromA, rook_sum_vec))
print("W_rook @ x equals rook_avg_vec? ", np.allclose(rook_avg_vec_fromW, rook_avg_vec))

print("\nVerification (Queen):")
print("A_queen @ x equals queen_sum_vec? ", np.allclose(queen_sum_vec_fromA, queen_sum_vec))
print("W_queen @ x equals queen_avg_vec? ", np.allclose(queen_avg_vec_fromW, queen_avg_vec))

# Show the raster results clearly
print("\nToy raster X (values 1..16):\n", X)
print("\nRook neighbor SUM (grid):\n", rook_sum_grid)
print("\nRook neighbor AVERAGE (grid)  == reshape(W_rook @ x):\n", rook_avg_grid)
print("\nQueen neighbor SUM (grid):\n", queen_sum_grid)
print("\nQueen neighbor AVERAGE (grid) == reshape(W_queen @ x):\n", queen_avg_grid)



A_rook (adjacency, 4-neighbors) (shape=(16, 16)):
  0   1   0   0   1   0   0   0   0   0   0   0   0   0   0   0
  1   0   1   0   0   1   0   0   0   0   0   0   0   0   0   0
  0   1   0   1   0   0   1   0   0   0   0   0   0   0   0   0
  0   0   1   0   0   0   0   1   0   0   0   0   0   0   0   0
  1   0   0   0   0   1   0   0   1   0   0   0   0   0   0   0
  0   1   0   0   1   0   1   0   0   1   0   0   0   0   0   0
  0   0   1   0   0   1   0   1   0   0   1   0   0   0   0   0
  0   0   0   1   0   0   1   0   0   0   0   1   0   0   0   0
  0   0   0   0   1   0   0   0   0   1   0   0   1   0   0   0
  0   0   0   0   0   1   0   0   1   0   1   0   0   1   0   0
  0   0   0   0   0   0   1   0   0   1   0   1   0   0   1   0
  0   0   0   0   0   0   0   1   0   0   1   0   0   0   0   1
  0   0   0   0   0   0   0   0   1   0   0   0   0   1   0   0
  0   0   0   0   0   0   0   0   0   1   0   0   1   0   1   0
  0   0   0   0   0   0   0   0   0   0   1   0   0  

### Tiny 4×4 Rook & Queen: Linear-Algebra Summary

We index an $m\times n$ grid (here $m=n=4$, so $N=mn=16$).  
Stack values row-major: $x=\mathrm{vec}(X)\in\mathbb{R}^{N}$.

**1D chain adjacencies (no wrap):**
- $C_m\in\mathbb{R}^{m\times m}$ on rows, $C_n\in\mathbb{R}^{n\times n}$ on columns.
- Identities $I_m, I_n$.

**Rook (4-neighbors) adjacency**
$$
A_{\text{rook}} \;=\; I_n\otimes C_m \;+\; C_n\otimes I_m.
$$

**Queen (8-neighbors) adjacency**
$$
A_{\text{queen}} \;=\; A_{\text{rook}} \;+\; C_n\otimes C_m.
$$

**Degrees, weights, Laplacians**
$$
D \;=\; \operatorname{diag}(A\mathbf{1}), \qquad
W \;=\; D^{-1}A, \qquad
L \;=\; D - A.
$$

> Rook degrees on a rectangle: corners $=2$, edges $=3$, interior $=4$.  
> Queen degrees: corners $=3$, edges $=5$, interior $=8$.

---

#### Stencil equivalences on a raster $X\in\mathbb{R}^{m\times n}$

**Rook — neighbor *sum* (matrix form)**  
Let $x=\mathrm{vec}(X)$.
$$
A_{\text{rook}}\,x \;=\; \mathrm{vec}\!\big(C_m X \;+\; X C_n^\top\big).
$$

**Rook — neighbor *average* (row-standardized)**  
Let $d^{\text{rook}} = A_{\text{rook}}\mathbf{1}\in\mathbb{R}^{N}$ and reshape it to $m\times n$.
Using elementwise division $\oslash$,
$$
W_{\text{rook}}\,x \;=\;
\mathrm{vec}\!\left(\big(C_m X + X C_n^\top\big)\;\oslash\;\operatorname{reshape}\!\big(d^{\text{rook}},\,m,n\big)\right).
$$

**Queen — neighbor *sum***
$$
A_{\text{queen}}\,x \;=\;
\mathrm{vec}\!\big(C_m X \;+\; X C_n^\top \;+\; C_m X C_n^\top\big).
$$

**Queen — neighbor *average* (row-standardized)**  
Let $d^{\text{queen}} = A_{\text{queen}}\mathbf{1}$.
$$
W_{\text{queen}}\,x \;=\;
\mathrm{vec}\!\left(\big(C_m X + X C_n^\top + C_m X C_n^\top\big)\;\oslash\;\operatorname{reshape}\!\big(d^{\text{queen}},\,m,n\big)\right).
$$

---

**Notes**
- $\otimes$ is the Kronecker product; $\mathrm{vec}(\cdot)$ stacks columns (or rows—use a consistent convention across code and math).
- The equalities show that multiplying by $A$ (or $W$) matches applying the corresponding **rook/queen stencil** and (optionally) averaging by the local degree.
